# 01 - Entity Resolution Exploratory Data Analysis

This notebook is read-only with respect to the raw dataset. It discovers train/test TSV files, uses the production loader, measures data quality and ground-truth structure, creates analytical plots, and writes reproducible reports under `artifacts/reports/`.

In [5]:
from pathlib import Path
import json
import re
import sys
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if not SRC.exists():
    SRC = PROJECT_ROOT / 'code' / 'business_entity_resolution' / 'src'
sys.path.insert(0, str(SRC.resolve()))
from config import default_config
from data_loader import DataLoadingError, load_table

CONFIG = default_config().resolved(PROJECT_ROOT)
REPORT_DIR = PROJECT_ROOT / 'artifacts' / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Data root:', CONFIG.dataset.data_root)
print('Report directory:', REPORT_DIR)

ModuleNotFoundError: No module named 'config'

In [ ]:
# Discover actual TSV files rather than assuming a fixed source count.
train_files = sorted((CONFIG.dataset.data_root / CONFIG.dataset.train_dir).glob('*.tsv'))
test_files = sorted((CONFIG.dataset.data_root / CONFIG.dataset.test_dir).glob('*.tsv'))
if not train_files and not test_files:
    raise FileNotFoundError(f'No TSV files found below {CONFIG.dataset.data_root}')
discovered = {'train': [str(p) for p in train_files], 'test': [str(p) for p in test_files]}
display(pd.DataFrame([(split, path) for split, paths in discovered.items() for path in paths], columns=['split', 'path']))
print('Configured train files:', CONFIG.dataset.train_source_files)
print('Configured test files:', CONFIG.dataset.test_source_files)

In [ ]:
def load_discovered(path, table_name):
    # The production loader enforces UTF-8, sep='\t', string-safe IDs,
    # non-empty files, malformed-row errors, and schema validation.
    required = [CONFIG.schema.entity_id_column, CONFIG.schema.name_column, CONFIG.schema.address_column, CONFIG.schema.country_column]
    return load_table(path, required_columns=required, table_name=table_name, dtype={c: 'string' for c in required}, separator='\t')

tables = {}
for split, paths in discovered.items():
    for path_text in paths:
        path = Path(path_text)
        try:
            tables[f'{split}/{path.name}'] = load_discovered(path, f'{split}/{path.name}')
        except DataLoadingError as exc:
            print('Rejected:', exc)

dimensions = pd.DataFrame([{'table': name, 'rows': len(frame), 'columns': frame.shape[1], 'column_names': list(frame.columns)} for name, frame in tables.items()])
display(dimensions)

In [ ]:
def quality_summary(frame, table_name):
    id_col = CONFIG.schema.entity_id_column
    name_col = CONFIG.schema.name_column
    address_col = CONFIG.schema.address_column
    country_col = CONFIG.schema.country_column
    out = {'table': table_name, 'rows': int(len(frame)), 'columns': int(frame.shape[1]), 'duplicate_entity_ids': int(frame[id_col].duplicated().sum())}
    for col in frame.columns:
        values = frame[col].astype('string')
        out[f'{col}.dtype'] = str(frame[col].dtype)
        out[f'{col}.missing'] = int(frame[col].isna().sum())
        out[f'{col}.empty'] = int(values.fillna('').eq('').sum())
        out[f'{col}.unique'] = int(values.nunique(dropna=False))
    for col in [name_col, address_col]:
        lengths = frame[col].fillna('').astype(str).str.len()
        tokens = frame[col].fillna('').astype(str).str.findall(r'[\w]+').str.len()
        out[f'{col}.length_mean'] = float(lengths.mean())
        out[f'{col}.length_median'] = float(lengths.median())
        out[f'{col}.length_p95'] = float(lengths.quantile(.95))
        out[f'{col}.token_mean'] = float(tokens.mean())
        out[f'{col}.token_median'] = float(tokens.median())
    out['duplicate_full_records'] = int(frame.duplicated().sum())
    return out

quality = pd.DataFrame([quality_summary(frame, name) for name, frame in tables.items()])
display(quality.T)

In [ ]:
record_tables = {name: frame for name, frame in tables.items() if 'ground_truth' not in name}
country_rows = []
for name, frame in record_tables.items():
    counts = frame[CONFIG.schema.country_column].fillna('').astype(str).value_counts(dropna=False)
    country_rows.extend({'table': name, 'country': str(country), 'count': int(count)} for country, count in counts.items())
country_distribution = pd.DataFrame(country_rows)
display(country_distribution.sort_values(['table', 'count'], ascending=[True, False]))

# Data-quality pattern counts are measured, not interpreted as labels.
pattern_rows = []
for name, frame in record_tables.items():
    for column in [CONFIG.schema.name_column, CONFIG.schema.address_column, CONFIG.schema.country_column]:
        values = frame[column].fillna('').astype(str)
        pattern_rows.append({'table': name, 'column': column, 'missing_or_empty': int(values.eq('').sum()), 'non_ascii': int(values.map(lambda x: any(ord(ch) > 127 for ch in x)).sum()), 'punctuation_present': int(values.str.contains(r'[^\w\s]', regex=True).sum())})
display(pd.DataFrame(pattern_rows))

In [ ]:
# Ground truth is loaded separately and used only for analysis.
gt_path = CONFIG.dataset.ground_truth_path()
ground_truth = load_table(gt_path, required_columns=[CONFIG.schema.ground_truth_source_id_column, CONFIG.schema.ground_truth_matches_column], table_name='ground truth', dtype={CONFIG.schema.ground_truth_source_id_column: 'string', CONFIG.schema.ground_truth_matches_column: 'string'}, separator='\t')
match_col = CONFIG.schema.ground_truth_matches_column
ground_truth['match_ids'] = ground_truth[match_col].fillna('').astype(str).map(lambda x: [v for v in x.split(',') if v])
ground_truth['match_count'] = ground_truth['match_ids'].str.len()
ground_truth['s2_count'] = ground_truth['match_ids'].map(lambda xs: sum(str(x).startswith('S2-') for x in xs))
ground_truth['s3_count'] = ground_truth['match_ids'].map(lambda xs: sum(str(x).startswith('S3-') for x in xs))
ground_truth['is_singleton'] = ground_truth['match_count'].eq(0)
ground_truth['has_s2_and_s3'] = ground_truth['s2_count'].gt(0) & ground_truth['s3_count'].gt(0)
match_distribution = ground_truth['match_count'].value_counts().sort_index().rename_axis('match_count').reset_index(name='reference_count')
overlap = pd.DataFrame({'pattern': ['zero_matches', 's2_only', 's3_only', 's2_and_s3'], 'reference_count': [int(ground_truth['match_count'].eq(0).sum()), int((ground_truth['s2_count'].gt(0) & ground_truth['s3_count'].eq(0)).sum()), int((ground_truth['s3_count'].gt(0) & ground_truth['s2_count'].eq(0)).sum()), int(ground_truth['has_s2_and_s3'].sum())]})
display(match_distribution)
display(overlap)
print('Reference entities:', len(ground_truth))
print('Matched entities:', int(ground_truth['match_count'].gt(0).sum()))
print('Singleton proportion:', float(ground_truth['is_singleton'].mean()))
print('Multi-match proportion:', float(ground_truth['match_count'].gt(1).mean()))

In [ ]:
# Representative measured variation examples: repeated normalized-ish
# punctuation/case patterns are shown without altering raw data.
def representative_variations(frame, column, n=10):
    values = frame[column].fillna('').astype(str)
    normalized = values.str.casefold().str.replace(r'[^\w\s]', '', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()
    view = pd.DataFrame({'raw': values, 'simple_comparison_form': normalized})
    return view[(view['raw'] != '') & (view['raw'] != view['simple_comparison_form'])].drop_duplicates().head(n)
for name, frame in record_tables.items():
    print('NAME variations:', name); display(representative_variations(frame, CONFIG.schema.name_column))
    print('ADDRESS variations:', name); display(representative_variations(frame, CONFIG.schema.address_column))

In [ ]:
# Analytical plots only.
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
country_distribution.pivot_table(index='country', columns='table', values='count', aggfunc='sum', fill_value=0).plot.bar(ax=axes[0,0], title='Country distribution')
for name, frame in list(record_tables.items())[:2]:
    frame[CONFIG.schema.name_column].fillna('').astype(str).str.len().plot.hist(bins=30, alpha=.5, ax=axes[0,1], label=name)
axes[0,1].set_title('Business-name lengths'); axes[0,1].legend()
for name, frame in list(record_tables.items())[:2]:
    frame[CONFIG.schema.address_column].fillna('').astype(str).str.len().plot.hist(bins=30, alpha=.5, ax=axes[0,2], label=name)
axes[0,2].set_title('Address lengths'); axes[0,2].legend()
match_distribution.plot.bar(x='match_count', y='reference_count', legend=False, ax=axes[1,0], title='Matches per reference entity')
overlap.plot.bar(x='pattern', y='reference_count', legend=False, ax=axes[1,1], title='S2/S3 overlap')
missing = quality.set_index('table').filter(regex='missing_or_empty|\.empty$').T
missing.plot.bar(ax=axes[1,2], title='Missing/empty indicators')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'eda_plots.png', dpi=150)
plt.show()

In [ ]:
summary = {'discovered_files': discovered, 'dimensions': dimensions.to_dict(orient='records'), 'quality': quality.to_dict(orient='records'), 'country_distribution': country_distribution.to_dict(orient='records'), 'match_distribution': match_distribution.to_dict(orient='records'), 'source_overlap': overlap.to_dict(orient='records')}
(REPORT_DIR / 'eda_summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
quality.to_csv(REPORT_DIR / 'eda_summary.csv', index=False)

observations = [
    f'Dataset dimensions were measured for {len(tables)} loaded tables.',
    f'The ground truth contains {len(ground_truth)} reference rows; {int(ground_truth.match_count.eq(0).sum())} have zero matches.',
    f'{int(ground_truth.match_count.gt(1).sum())} reference rows have multiple matches.',
    f'S2-only, S3-only, and dual-source rows were measured as {overlap.to_dict(orient="records")}.',
    'Raw names and addresses were preserved; the examples above are descriptive views only.',
    'Potential leakage risk: ground truth must be restricted to training validation and never used for test inference.',
    'Blocking implication: exact and token keys should be tested against measured name/address variation and candidate recall.',
    'Feature implication: missingness, token overlap, character similarity, numeric/postal tokens, and source indicators are measurable candidate features.'
]
(REPORT_DIR / 'observations.md').write_text('# Measured EDA Observations\n\n' + '\n'.join(f'- {item}' for item in observations) + '\n', encoding='utf-8')
print('Wrote:', REPORT_DIR / 'eda_summary.json')
print('Wrote:', REPORT_DIR / 'eda_summary.csv')
print('Wrote:', REPORT_DIR / 'observations.md')

## Structured conclusion

1. **Dataset dimensions:** See `dimensions` and `eda_summary.json`; values are computed at execution time.
2. **Data-quality findings:** Missingness, empty strings, duplicate IDs, duplicate records, dtypes, and unique counts are measured above.
3. **Name noise findings:** Representative raw/comparison-form examples and name-length/token statistics are measured above.
4. **Address noise findings:** Address-length/token statistics, numeric/postal-like tokens, and representative examples guide blocking/features.
5. **Country findings:** Country distributions are measured without restricting the country vocabulary.
6. **Match-count findings:** Zero/one/multiple match distributions are computed from training ground truth only.
7. **Singleton findings:** Singleton/no-match counts and proportions are measured and included in plots.
8. **Blocking implications:** Compare complementary keys using candidate recall and reduction ratio; do not rely on a single exact key.
9. **Feature-engineering implications:** Preserve missingness, token, character, numeric/postal, locality-like, country, and source signals for validation.
10. **Potential leakage risks:** Do not use ground-truth columns, fitted vocabularies, or validation-derived decisions during test inference.